# M1 — Lexical retrieval on Kaggle: E01–E04

**Purpose.** Run the complete CPU-heavy M1 lexical experiment on Kaggle rather than a local Jupyter kernel. It reproduces the frozen M0 proxy, logs a live ClearML Task, selects the best text preprocessing and writes reusable artifacts to `/kaggle/working/avito-m1-lexical/`.

**Success criterion.** The notebook finishes with: a ClearML task link, `validation_results.csv`, a manifest, prepared text Parquet files, and top-200 candidate lists for validation and benchmark queries.

**Safety.** Input files are read only. Secret values are never displayed or written to an artifact.

## Plan

1. **Kaggle environment and dependencies** — verify Kaggle, install only absent open-source packages, locate the attached dataset and create a writable output directory.
2. **ClearML secrets** — load two credentials from Kaggle Secrets and create a live experiment task.
3. **Frozen validation protocol** — reproduce the M0 group-disjoint proxy and category rule exactly.
4. **E01–E03 lexical control** — build the original BM25, field ablations, title char-TFIDF and RRF.
5. **E04 preprocessing** — compare Russian stemming, lemmatization and stop-word configurations under identical conditions.
6. **Artifacts and candidate exports** — save selected text representations and final top-200 candidate lists under `/kaggle/working`.
7. **Results and decision** — log metrics, runtime and the chosen configuration to ClearML.

Run the notebook through **Save Version → Save & Run All**. It must run from a clean kernel; do not rely on variables from interactive cells.

## 1. Kaggle environment, dependencies and input data

Attach one private Kaggle Dataset containing `train.parquet`, `benchmark_queries.parquet` and `benchmark_items.parquet` in one directory. Turn **Internet on** for this experiment: ClearML needs network access, and the first run may need to install lightweight open-source packages. The code finds the input directory by filenames rather than assuming a Dataset slug.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "clearml": "clearml>=1.16",
    "rank_bm25": "rank-bm25>=0.2.2",
    "pymorphy3": "pymorphy3>=2.0",
    "snowballstemmer": "snowballstemmer>=2.2",
    "stop_words": "stop-words>=2018.7.23",
}
missing_packages = [requirement for module, requirement in PACKAGE_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-input", *missing_packages])

print({"installed_now": missing_packages, "python": sys.version.split()[0]})

In [ ]:
import gc
import json
import os
import re
import resource
import shutil
import sys
import time
from collections import Counter
from functools import lru_cache, partial
from pathlib import Path

import numpy as np
import pandas as pd
import pymorphy3
import snowballstemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from stop_words import get_stop_words

assert Path("/kaggle").exists(), "This notebook is intended for a Kaggle Notebook runtime."

SEED = 42
np.random.seed(SEED)
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/avito-m1-lexical")
ARTIFACT_DIR = OUTPUT_ROOT / "text_preprocessing" / "m1_lexical_best_v1"
TRAIN_FILENAME = "train.parquet"
BENCHMARK_QUERIES_FILENAME = "benchmark_queries.parquet"
BENCHMARK_ITEMS_FILENAME = "benchmark_items.parquet"


def locate_dataset_dir() -> Path:
    explicit = os.environ.get("AVITO_KAGGLE_DATA_DIR")
    if explicit:
        candidate = Path(explicit)
        required = (candidate / TRAIN_FILENAME, candidate / BENCHMARK_QUERIES_FILENAME, candidate / BENCHMARK_ITEMS_FILENAME)
        if all(path.is_file() for path in required):
            return candidate
        raise FileNotFoundError(f"AVITO_KAGGLE_DATA_DIR={candidate} does not contain all three required Parquet files.")

    matches = []
    for train_path in KAGGLE_INPUT_ROOT.rglob(TRAIN_FILENAME):
        parent = train_path.parent
        required = (parent / BENCHMARK_QUERIES_FILENAME, parent / BENCHMARK_ITEMS_FILENAME)
        if all(path.is_file() for path in required):
            matches.append(parent)
    if len(matches) != 1:
        raise FileNotFoundError(
            "Attach exactly one Dataset that contains train.parquet, benchmark_queries.parquet and "
            f"benchmark_items.parquet in one folder. Found candidates: {[str(path) for path in matches]}"
        )
    return matches[0]


DATA_DIR = locate_dataset_dir()
TRAIN_PATH = DATA_DIR / TRAIN_FILENAME
BENCHMARK_QUERIES_PATH = DATA_DIR / BENCHMARK_QUERIES_FILENAME
BENCHMARK_ITEMS_PATH = DATA_DIR / BENCHMARK_ITEMS_FILENAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOP_K = 200
METRIC_KS = (1, 5, 10, 20, 50, 200)
BM25_K1 = 1.5
BM25_B = 0.75
CHAR_MAX_FEATURES = 200_000
CHAR_BATCH_SIZE = 32
PREPROCESSING_VERSION = "m1_lexical_best_v1"
PREPROCESSING_ORDER = (
    "control",
    "stem",
    "lemma",
    "stopwords",
    "stem_stopwords",
    "lemma_stopwords",
)
NOTEBOOK_STARTED = time.perf_counter()

print({
    "data_dir": str(DATA_DIR),
    "input_bytes": {path.name: path.stat().st_size for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)},
    "output_root": str(OUTPUT_ROOT),
    "top_k": TOP_K,
    "preprocessing_variants": list(PREPROCESSING_ORDER),
})

## 2. Live ClearML task via Kaggle Secrets

In the Kaggle Notebook editor, create two secrets in **Add-ons → Secrets** with labels `CLEARML_API_ACCESS_KEY` and `CLEARML_API_SECRET_KEY`, then enable both for this notebook. The ClearML hosts are public endpoints and are set in code; only the two credentials are secrets.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets_client = UserSecretsClient()
for secret_name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
    try:
        value = secrets_client.get_secret(secret_name)
    except Exception as exc:
        raise RuntimeError(
            f"Kaggle Secret '{secret_name}' is unavailable. Add it in Add-ons → Secrets and enable it for this notebook."
        ) from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret '{secret_name}' is empty.")
    os.environ[secret_name] = value

# These endpoints are not credentials. Environment variables allow an override
# without changing the notebook if the ClearML deployment changes later.
os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
os.environ.pop("CLEARML_OFFLINE_MODE", None)

from clearml import Task

CLEARML_PROJECT = "avito-retrieval"
CLEARML_TASK_NAME = "E01-E04__lexical-preprocessing__kaggle__s42"
clearml_task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
clearml_task.connect(
    {
        "stage": "M1_lexical_retrieval_preprocessing_kaggle",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "bm25_k1": BM25_K1,
        "bm25_b": BM25_B,
        "category_partition": "item_category_id == search_category; full-corpus fallback when no matching corpus partition exists",
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": CHAR_MAX_FEATURES},
        "preprocessing_variants": list(PREPROCESSING_ORDER),
        "input_dir": str(DATA_DIR),
        "output_dir": str(OUTPUT_ROOT),
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_project": CLEARML_PROJECT, "clearml_task_id": clearml_task.id, "credentials_loaded": True})

## 2. Recreate the frozen M0 proxy exactly

`query_group` is built from all search-side fields over the concatenation of train and benchmark query contexts. The split is then applied to the benchmark-corpus-aligned train positive rows, exactly as in M0.

In [ ]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
BENCHMARK_QUERY_COLUMNS = ["query_id", *SEARCH_COLUMNS]
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
    "item_category_id",
]

load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=BENCHMARK_QUERY_COLUMNS)
benchmark_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS)
load_seconds = time.perf_counter() - load_started


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

all_contexts = pd.concat(
    [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
    ignore_index=True,
)
all_group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = all_group_ids[: len(train_pairs)]

benchmark_item_id_set = set(benchmark_items["item_id"].astype(str))
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
proxy_train_idx, proxy_valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
proxy_train_pairs = proxy_pairs.iloc[proxy_train_idx].copy()
proxy_valid_pairs = proxy_pairs.iloc[proxy_valid_idx].copy()
assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))

validation_queries = (
    proxy_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
gold_by_group = (
    proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]
    .agg(lambda values: frozenset(values.astype(str)))
    .to_dict()
)
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

proxy_summary = pd.DataFrame(
    [
        {"partition": "train", "positive_rows": len(proxy_train_pairs), "query_groups": proxy_train_pairs["query_group"].nunique()},
        {"partition": "validation", "positive_rows": len(proxy_valid_pairs), "query_groups": proxy_valid_pairs["query_group"].nunique()},
    ]
)
display(proxy_summary)
print({
    "load_seconds": round(load_seconds, 2),
    "proxy_positive_rows": len(proxy_pairs),
    "validation_unique_queries": len(validation_queries),
    "validation_gold_cardinality_median": float(np.median([len(gold) for gold in gold_sets])),
})

In [ ]:
# M0 hard partition, with the documented fallback if a category has no corpus rows.
# In this split, 5,309 queries have category 114; one query has category 0,
# which has no matching item_category_id and therefore uses the full corpus.
candidate_items = benchmark_items.reset_index(drop=True)
candidate_item_ids = candidate_items["item_id"].astype(str).to_numpy()
all_candidate_indices = np.arange(len(candidate_items), dtype=np.int64)
category_to_indices = {
    category: group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}


def allowed_indices_for_category(category: object) -> tuple[np.ndarray, bool]:
    indices = category_to_indices.get(category)
    if indices is None or len(indices) == 0:
        return all_candidate_indices, True
    return indices, False


allowed_indices_by_query = []
used_full_corpus_fallback = []
for category in validation_queries["search_category"]:
    allowed, fallback = allowed_indices_for_category(category)
    allowed_indices_by_query.append(allowed)
    used_full_corpus_fallback.append(fallback)

category_oracle_recall = float(
    np.mean([
        len(gold & set(candidate_item_ids[allowed])) / len(gold)
        for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)
    ])
)
assert category_oracle_recall == 1.0, "The category rule plus fallback removes a validation positive."

print({
    "validation_categories": validation_queries["search_category"].value_counts().to_dict(),
    "items_in_category_114": int((candidate_items["item_category_id"] == 114).sum()),
    "full_benchmark_items": len(candidate_items),
    "full_corpus_fallback_queries": int(sum(used_full_corpus_fallback)),
    "validation_category_oracle_recall@50": category_oracle_recall,
})

## 3. Text preprocessing and exact sparse BM25

Every transformation starts from the same control normalization: lower case, `ё → е`, non-alphanumeric characters replaced by spaces, and collapsed whitespace. E04 then applies one of: Russian Snowball stemming, `pymorphy3` lemmatization, the `stop-words` Russian list, or a combination. Stop words are removed before morphology.

The sparse implementation is validated against `rank_bm25.BM25Okapi` on a deterministic example. All preprocessing is performed locally; no external API is used.

In [ ]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")


def normalize_russian_text(value: object) -> str:
    """M1 control transformation shared by all preprocessing variants."""
    text = "" if pd.isna(value) else str(value)
    text = text.lower().replace("ё", "е")
    text = NON_WORD_RE.sub(" ", text)
    return " ".join(text.split())


RUSSIAN_STOPWORDS = frozenset(
    token
    for word in get_stop_words("ru")
    for token in normalize_russian_text(word).split()
)
STEMMER = snowballstemmer.stemmer("russian")
MORPH = pymorphy3.MorphAnalyzer()

PREPROCESSING_SPECS = {
    "control": {"morphology": "none", "remove_stopwords": False},
    "stem": {"morphology": "snowball_russian", "remove_stopwords": False},
    "lemma": {"morphology": "pymorphy3_normal_form", "remove_stopwords": False},
    "stopwords": {"morphology": "none", "remove_stopwords": True},
    "stem_stopwords": {"morphology": "snowball_russian", "remove_stopwords": True},
    "lemma_stopwords": {"morphology": "pymorphy3_normal_form", "remove_stopwords": True},
}
assert tuple(PREPROCESSING_SPECS) == PREPROCESSING_ORDER


@lru_cache(maxsize=1_000_000)
def stem_token(token: str) -> str:
    return STEMMER.stemWord(token)


@lru_cache(maxsize=1_000_000)
def lemma_token(token: str) -> str:
    parsed = MORPH.parse(token)
    return parsed[0].normal_form if parsed else token


def preprocess_russian_text(value: object, preprocessing_key: str = "control") -> str:
    """Apply one deterministic E04 configuration to a text value."""
    spec = PREPROCESSING_SPECS[preprocessing_key]
    tokens = normalize_russian_text(value).split()
    if spec["remove_stopwords"]:
        tokens = [token for token in tokens if token not in RUSSIAN_STOPWORDS]
    if spec["morphology"] == "snowball_russian":
        tokens = [stem_token(token) for token in tokens]
    elif spec["morphology"] == "pymorphy3_normal_form":
        tokens = [lemma_token(token) for token in tokens]
    return " ".join(tokens)


class SparseBM25:
    """Memory-efficient exact Okapi BM25 over a CountVectorizer CSC matrix."""

    def __init__(self, preprocessing_key: str = "control", k1: float = 1.5, b: float = 0.75, epsilon: float = 0.25):
        self.preprocessing_key = preprocessing_key
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(
            preprocessor=partial(preprocess_russian_text, preprocessing_key=self.preprocessing_key),
            token_pattern=TOKEN_PATTERN,
            lowercase=False,
            dtype=np.float32,
        )
        counts_csr = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts_csr.sum(axis=1)).ravel().astype(np.float32)
        self.n_docs = counts_csr.shape[0]
        self.avgdl = float(self.doc_len.mean())
        self.matrix = counts_csr.tocsc()
        del counts_csr

        document_frequency = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        average_idf = float(idf.mean())
        idf[idf < 0] = self.epsilon * average_idf
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)
        self.analyzer = self.vectorizer.build_analyzer()
        return self

    @property
    def n_features(self) -> int:
        return len(self.vectorizer.vocabulary_)

    def score(self, query: str) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(self.analyzer(query)).items():
            feature_idx = self.vectorizer.vocabulary_.get(token)
            if feature_idx is None:
                continue
            start, stop = self.matrix.indptr[feature_idx : feature_idx + 2]
            rows = self.matrix.indices[start:stop]
            term_tf = self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature_idx] * (
                term_tf * (self.k1 + 1.0) / (term_tf + self.norm[rows])
            )
        return scores

    def top_k(self, query: str, k: int, allowed_indices: np.ndarray | None = None) -> np.ndarray:
        scores = self.score(query)
        allowed = np.arange(self.n_docs, dtype=np.int64) if allowed_indices is None else allowed_indices
        k = min(k, len(allowed))
        eligible_scores = scores[allowed]
        selected_positions = np.argpartition(eligible_scores, len(allowed) - k)[len(allowed) - k :]
        selected_positions = selected_positions[np.argsort(eligible_scores[selected_positions])[::-1]]
        return allowed[selected_positions]


# Correctness check against the open-source reference implementation.
toy_documents = ["ремонт ноутбука", "ремонт телевизора", "установка двери"]
toy_query = "ремонт"
reference_scores = BM25Okapi([document.split() for document in toy_documents], k1=BM25_K1, b=BM25_B).get_scores(toy_query.split())
checked_scores = SparseBM25(k1=BM25_K1, b=BM25_B).fit(toy_documents).score(toy_query)
np.testing.assert_allclose(checked_scores, reference_scores, rtol=1e-6, atol=1e-6)
assert preprocess_russian_text("Ёлки, и баня!", "control") == "елки и баня"
assert preprocess_russian_text("Ёлки, и баня!", "stopwords") == "елки баня"
print({"bm25_reference_check": "passed", "russian_stopwords": len(RUSSIAN_STOPWORDS), "preprocessing_configs": PREPROCESSING_SPECS})

In [ ]:
def compose_document_text(frame: pd.DataFrame, fields: tuple[str, ...]) -> list[str]:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text.tolist()


def retrieve_bm25(index: SparseBM25, queries: list[str], allowed_by_query: list[np.ndarray], top_k: int) -> tuple[list[np.ndarray], float]:
    started = time.perf_counter()
    rankings = [
        index.top_k(query, top_k, allowed)
        for query, allowed in zip(queries, allowed_by_query, strict=True)
    ]
    return rankings, time.perf_counter() - started


def top_k_from_scores(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    selected = np.argpartition(scores, len(scores) - k)[len(scores) - k :]
    return selected[np.argsort(scores[selected])[::-1]]


def macro_recall(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    recalls = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        predicted = set(candidate_item_ids[ranking[:k]])
        recalls.append(len(predicted & relevant) / len(relevant))
    return float(np.mean(recalls))


def hit_rate(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    hits = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        hits.append(bool(set(candidate_item_ids[ranking[:k]]) & relevant))
    return float(np.mean(hits))


def peak_rss_mb() -> float:
    max_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return float(max_rss / (1024 * 1024 if sys.platform == "darwin" else 1024))


def evaluate(experiment: str, rankings: list[np.ndarray], *, index_seconds: float, retrieval_seconds: float,
             n_features: int, representation: str, metadata: dict[str, object] | None = None) -> dict[str, object]:
    record: dict[str, object] = {
        "experiment": experiment,
        "representation": representation,
        "index_seconds": index_seconds,
        "retrieval_seconds": retrieval_seconds,
        "total_seconds": index_seconds + retrieval_seconds,
        "features": n_features,
        "peak_rss_mb": peak_rss_mb(),
    }
    for k in METRIC_KS:
        record[f"recall@{k}"] = macro_recall(rankings, gold_sets, k)
    record["hit_rate@50"] = hit_rate(rankings, gold_sets, 50)
    if metadata:
        record.update(metadata)
    return record


def fit_sparse_bm25(documents: list[str], preprocessing_key: str = "control") -> tuple[SparseBM25, float]:
    started = time.perf_counter()
    index = SparseBM25(preprocessing_key=preprocessing_key, k1=BM25_K1, b=BM25_B).fit(documents)
    return index, time.perf_counter() - started


def fit_and_retrieve_char_tfidf(
    documents: list[str], queries: list[str], allowed_by_query: list[np.ndarray], preprocessing_key: str,
) -> tuple[TfidfVectorizer, list[np.ndarray], float, float]:
    """Batch sparse cosine retrieval to make the preprocessing comparison feasible."""
    fit_started = time.perf_counter()
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        preprocessor=partial(preprocess_russian_text, preprocessing_key=preprocessing_key),
        lowercase=False,
        ngram_range=(3, 5),
        min_df=2,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )
    matrix = vectorizer.fit_transform(documents)
    index_seconds = time.perf_counter() - fit_started

    retrieval_started = time.perf_counter()
    query_matrix = vectorizer.transform(queries)
    rankings: list[np.ndarray] = []
    for start in range(0, query_matrix.shape[0], CHAR_BATCH_SIZE):
        stop = min(start + CHAR_BATCH_SIZE, query_matrix.shape[0])
        batch_scores = (query_matrix[start:stop] @ matrix.T).toarray()
        for row_scores, allowed in zip(batch_scores, allowed_by_query[start:stop], strict=True):
            selected_positions = top_k_from_scores(row_scores[allowed], TOP_K)
            rankings.append(allowed[selected_positions])
    retrieval_seconds = time.perf_counter() - retrieval_started
    return vectorizer, rankings, index_seconds, retrieval_seconds


def rrf_fuse(rankings_by_source: list[np.ndarray], top_k: int, rrf_k: int = 60) -> np.ndarray:
    scores: dict[int, float] = {}
    for rankings in rankings_by_source:
        for rank, item_idx in enumerate(rankings, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (rrf_k + rank)
    ordered = sorted(scores, key=lambda item_idx: (-scores[item_idx], item_idx))
    return np.asarray(ordered[:top_k], dtype=np.int64)


query_text = validation_queries["search_query"].fillna("").astype(str).tolist()
query_with_filters = (
    validation_queries["search_query"].fillna("").astype(str)
    + " "
    + validation_queries["search_infm_params_text"].fillna("").astype(str)
).tolist()

assert len(query_text) == len(gold_sets) == 5310
results: list[dict[str, object]] = []
print({"validation_queries": len(query_text), "retrieval_top_k": TOP_K, "char_batch_size": CHAR_BATCH_SIZE})

## 4. E01 — Plain BM25

The prescribed first baseline indexes the concatenation of title, parameters and description; the query contains only `search_query`.

In [ ]:
all_fields = ("item_title_raw", "item_infm_params_text", "item_description_raw")
all_documents = compose_document_text(candidate_items, all_fields)
all_bm25, all_index_seconds = fit_sparse_bm25(all_documents, preprocessing_key="control")
rankings_all, all_retrieval_seconds = retrieve_bm25(all_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E01_bm25_title_params_description__query",
    rankings_all,
    index_seconds=all_index_seconds,
    retrieval_seconds=all_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

## 5. E02 — Controlled lexical ablations

Only one textual representation changes at a time. The all-fields index is reused for the query-filter ablation, so its retrieval timing excludes a duplicated index build.

In [ ]:
# E02a: title only.
title_documents = compose_document_text(candidate_items, ("item_title_raw",))
title_bm25, title_index_seconds = fit_sparse_bm25(title_documents, preprocessing_key="control")
rankings_title, title_retrieval_seconds = retrieve_bm25(title_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02a_bm25_title__query",
    rankings_title,
    index_seconds=title_index_seconds,
    retrieval_seconds=title_retrieval_seconds,
    n_features=title_bm25.n_features,
    representation="title | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])
del title_bm25, title_documents
gc.collect()

# E02b: title + item parameters.
title_params_documents = compose_document_text(candidate_items, ("item_title_raw", "item_infm_params_text"))
title_params_bm25, title_params_index_seconds = fit_sparse_bm25(title_params_documents, preprocessing_key="control")
rankings_title_params, title_params_retrieval_seconds = retrieve_bm25(title_params_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02b_bm25_title_params__query",
    rankings_title_params,
    index_seconds=title_params_index_seconds,
    retrieval_seconds=title_params_retrieval_seconds,
    n_features=title_params_bm25.n_features,
    representation="title + parameters | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])
del title_params_bm25, title_params_documents
gc.collect()

# E02c: add the search filter text to the E01 query representation.
rankings_all_filters, all_filters_retrieval_seconds = retrieve_bm25(all_bm25, query_with_filters, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02c_bm25_title_params_description__query_filters",
    rankings_all_filters,
    index_seconds=0.0,
    retrieval_seconds=all_filters_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query + search filters (reused E01 index)",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

## 6. E03 — Strong lexical alternatives and fusion

The control char-TFIDF representation is retained as the reference because character n-grams already encode much of Russian inflectional similarity. Its batched implementation preserves exact cosine scores while avoiding a Python-level transform loop for every query.

In [ ]:
# E03a: char n-gram TF-IDF on short titles with control preprocessing.
char_documents = compose_document_text(candidate_items, ("item_title_raw",))
char_vectorizer, rankings_char, char_index_seconds, char_retrieval_seconds = fit_and_retrieve_char_tfidf(
    char_documents, query_text, allowed_indices_by_query, preprocessing_key="control"
)
results.append(evaluate(
    "E03a_char_tfidf_title__query",
    rankings_char,
    index_seconds=char_index_seconds,
    retrieval_seconds=char_retrieval_seconds,
    n_features=len(char_vectorizer.vocabulary_),
    representation="char_wb 3–5 grams: title | query",
    metadata={"bm25_preprocessing": "not_applicable", "char_preprocessing": "control"},
))
print(results[-1])

# E03b: reciprocal-rank fusion of title-only and E01 all-fields BM25.
rrf_bm25_started = time.perf_counter()
rankings_rrf_bm25 = [rrf_fuse([title_ranking, all_ranking], TOP_K) for title_ranking, all_ranking in zip(rankings_title, rankings_all, strict=True)]
rrf_bm25_seconds = time.perf_counter() - rrf_bm25_started
results.append(evaluate(
    "E03b_rrf_bm25_title_all_fields",
    rankings_rrf_bm25,
    index_seconds=0.0,
    retrieval_seconds=rrf_bm25_seconds,
    n_features=0,
    representation="RRF(title-BM25, all-fields-BM25); reused indices",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

# E03c: fuse the strongest broad BM25 source with char TF-IDF.
rrf_char_started = time.perf_counter()
rankings_rrf_char = [rrf_fuse([all_ranking, char_ranking], TOP_K) for all_ranking, char_ranking in zip(rankings_all, rankings_char, strict=True)]
rrf_char_seconds = time.perf_counter() - rrf_char_started
results.append(evaluate(
    "E03c_rrf_bm25_all_fields_char_tfidf",
    rankings_rrf_char,
    index_seconds=0.0,
    retrieval_seconds=rrf_char_seconds,
    n_features=0,
    representation="RRF(all-fields-BM25, title-char-TFIDF); reused indices",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "control"},
))
print(results[-1])

## 7. E04 — Russian preprocessing ablation

The six configurations are compared first on the broad all-field BM25 source. Only the winner is then evaluated in the expensive char-TFIDF and RRF final-pipeline checks. This prevents us from selecting a morphology rule merely because it has a plausible linguistic motivation.

In [ ]:
preprocessing_rows: list[dict[str, object]] = []
preprocessing_rankings: dict[str, list[np.ndarray]] = {}

# E01/E03 already produced the control rankings. Their fitted matrices are no
# longer needed, so release them before sequentially fitting five new BM25 indices.
control_bm25_features = all_bm25.n_features
del all_bm25, char_vectorizer
gc.collect()

for preprocessing_key in PREPROCESSING_ORDER:
    if preprocessing_key == "control":
        rankings = rankings_all
        index_seconds = all_index_seconds
        retrieval_seconds = all_retrieval_seconds
        n_features = control_bm25_features
    else:
        index, index_seconds = fit_sparse_bm25(all_documents, preprocessing_key=preprocessing_key)
        rankings, retrieval_seconds = retrieve_bm25(index, query_text, allowed_indices_by_query, TOP_K)
        n_features = index.n_features

    record = evaluate(
        f"E04_bm25_all_fields__{preprocessing_key}",
        rankings,
        index_seconds=index_seconds,
        retrieval_seconds=retrieval_seconds,
        n_features=n_features,
        representation=f"title + parameters + description | query | preprocessing={preprocessing_key}",
        metadata={"bm25_preprocessing": preprocessing_key, "char_preprocessing": "not_applicable"},
    )
    results.append(record)
    preprocessing_rows.append(record)
    preprocessing_rankings[preprocessing_key] = rankings
    print({
        "preprocessing": preprocessing_key,
        "recall@50": round(float(record["recall@50"]), 6),
        "index_seconds": round(float(index_seconds), 2),
        "retrieval_seconds": round(float(retrieval_seconds), 2),
    })

    if preprocessing_key != "control":
        del index
        gc.collect()

preprocessing_frame = (
    pd.DataFrame(preprocessing_rows)
    .sort_values(["recall@50", "recall@200", "experiment"], ascending=[False, False, True])
    .reset_index(drop=True)
)
best_bm25_preprocessing = str(preprocessing_frame.iloc[0]["bm25_preprocessing"])
rankings_best_bm25 = preprocessing_rankings[best_bm25_preprocessing]
display(preprocessing_frame[["bm25_preprocessing", "recall@50", "recall@200", "hit_rate@50", "index_seconds", "retrieval_seconds", "features"]])
print({"best_bm25_preprocessing": best_bm25_preprocessing})

# Test whether the best word-level transformation helps the established char-TFIDF fusion.
rrf_best_bm25_started = time.perf_counter()
rankings_rrf_best_bm25_control_char = [
    rrf_fuse([bm25_ranking, char_ranking], TOP_K)
    for bm25_ranking, char_ranking in zip(rankings_best_bm25, rankings_char, strict=True)
]
rrf_best_bm25_control_char_seconds = time.perf_counter() - rrf_best_bm25_started
results.append(evaluate(
    "E04_rrf_best_bm25_control_char",
    rankings_rrf_best_bm25_control_char,
    index_seconds=0.0,
    retrieval_seconds=rrf_best_bm25_control_char_seconds,
    n_features=0,
    representation=f"RRF(best E04 BM25={best_bm25_preprocessing}, control title-char-TFIDF)",
    metadata={"bm25_preprocessing": best_bm25_preprocessing, "char_preprocessing": "control"},
))

# Also apply the same winning transformation to title char-TFIDF. If it loses,
# the final pipeline keeps the control char representation instead of forcing morphology.
if best_bm25_preprocessing == "control":
    rankings_best_char = rankings_char
    best_char_index_seconds = 0.0
    best_char_retrieval_seconds = 0.0
else:
    best_char_vectorizer, rankings_best_char, best_char_index_seconds, best_char_retrieval_seconds = fit_and_retrieve_char_tfidf(
        char_documents, query_text, allowed_indices_by_query, preprocessing_key=best_bm25_preprocessing
    )
    results.append(evaluate(
        f"E04_char_tfidf_title__{best_bm25_preprocessing}",
        rankings_best_char,
        index_seconds=best_char_index_seconds,
        retrieval_seconds=best_char_retrieval_seconds,
        n_features=len(best_char_vectorizer.vocabulary_),
        representation=f"char_wb 3–5 grams: title | query | preprocessing={best_bm25_preprocessing}",
        metadata={"bm25_preprocessing": "not_applicable", "char_preprocessing": best_bm25_preprocessing},
    ))
    del best_char_vectorizer
    gc.collect()

rrf_best_both_started = time.perf_counter()
rankings_rrf_best_both = [
    rrf_fuse([bm25_ranking, char_ranking], TOP_K)
    for bm25_ranking, char_ranking in zip(rankings_best_bm25, rankings_best_char, strict=True)
]
rrf_best_both_seconds = time.perf_counter() - rrf_best_both_started
results.append(evaluate(
    "E04_rrf_best_bm25_best_char",
    rankings_rrf_best_both,
    index_seconds=0.0,
    retrieval_seconds=rrf_best_both_seconds,
    n_features=0,
    representation=f"RRF(best E04 BM25={best_bm25_preprocessing}, char preprocessing={best_bm25_preprocessing})",
    metadata={"bm25_preprocessing": best_bm25_preprocessing, "char_preprocessing": best_bm25_preprocessing},
))

FINAL_PIPELINE_EXPERIMENTS = (
    "E03c_rrf_bm25_all_fields_char_tfidf",
    "E04_rrf_best_bm25_control_char",
    "E04_rrf_best_bm25_best_char",
)
final_candidates = [record for record in results if record["experiment"] in FINAL_PIPELINE_EXPERIMENTS]
best_final_result = sorted(
    final_candidates,
    key=lambda record: (-float(record["recall@50"]), -float(record["recall@200"]), str(record["experiment"])),
)[0]
print({
    "best_final_experiment": best_final_result["experiment"],
    "best_final_recall@50": round(float(best_final_result["recall@50"]), 6),
    "final_bm25_preprocessing": best_final_result["bm25_preprocessing"],
    "final_char_preprocessing": best_final_result["char_preprocessing"],
})

## 8. Persist selected text representations

The chosen representation is materialized once for future retrieval stages. The ignored local directory `artifacts/text_preprocessing/m1_lexical_best_v1/` contains field-level BM25 texts, title text for char-TFIDF, benchmark queries, train query contexts, deduplicated train-item texts, a manifest and a validation-results table. It contains no credentials.

In [ ]:
final_bm25_preprocessing = str(best_final_result["bm25_preprocessing"])
final_char_preprocessing = str(best_final_result["char_preprocessing"])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def preprocess_series(series: pd.Series, preprocessing_key: str) -> pd.Series:
    return series.fillna("").astype(str).map(
        lambda value: preprocess_russian_text(value, preprocessing_key=preprocessing_key)
    )


artifact_started = time.perf_counter()

artifact_items = candidate_items.loc[:, ["item_id", "item_category_id"]].copy()
for source_column, target_column in (
    ("item_title_raw", "item_title_bm25"),
    ("item_infm_params_text", "item_params_bm25"),
    ("item_description_raw", "item_description_bm25"),
):
    artifact_items[target_column] = preprocess_series(candidate_items[source_column], final_bm25_preprocessing)
artifact_items["item_text_bm25"] = (
    artifact_items["item_title_bm25"] + " "
    + artifact_items["item_params_bm25"] + " "
    + artifact_items["item_description_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_items["item_title_char_tfidf"] = preprocess_series(candidate_items["item_title_raw"], final_char_preprocessing)

artifact_benchmark_queries = benchmark_queries.loc[:, ["query_id", "search_category"]].copy()
artifact_benchmark_queries["search_query_bm25"] = preprocess_series(
    benchmark_queries["search_query"], final_bm25_preprocessing
)
artifact_benchmark_queries["search_filters_bm25"] = preprocess_series(
    benchmark_queries["search_infm_params_text"], final_bm25_preprocessing
)
artifact_benchmark_queries["search_text_bm25"] = (
    artifact_benchmark_queries["search_query_bm25"] + " " + artifact_benchmark_queries["search_filters_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_benchmark_queries["search_query_char_tfidf"] = preprocess_series(
    benchmark_queries["search_query"], final_char_preprocessing
)

train_query_contexts = (
    train_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
artifact_train_queries = train_query_contexts.loc[:, ["query_group", "search_category"]].copy()
artifact_train_queries["search_query_bm25"] = preprocess_series(
    train_query_contexts["search_query"], final_bm25_preprocessing
)
artifact_train_queries["search_filters_bm25"] = preprocess_series(
    train_query_contexts["search_infm_params_text"], final_bm25_preprocessing
)
artifact_train_queries["search_text_bm25"] = (
    artifact_train_queries["search_query_bm25"] + " " + artifact_train_queries["search_filters_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_train_queries["search_query_char_tfidf"] = preprocess_series(
    train_query_contexts["search_query"], final_char_preprocessing
)

# M4 needs the selected item's text as well. Store one deterministic copy per
# train item_id, rather than repeating the same listing for every clicked pair.
train_item_texts = pd.read_parquet(
    TRAIN_PATH,
    columns=["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw"],
)
train_item_texts = (
    train_item_texts.sort_values("item_id")
    .drop_duplicates("item_id", keep="first")
    .reset_index(drop=True)
)
assert train_item_texts["item_id"].is_unique
artifact_train_items = train_item_texts.loc[:, ["item_id"]].copy()
for source_column, target_column in (
    ("item_title_raw", "item_title_bm25"),
    ("item_infm_params_text", "item_params_bm25"),
    ("item_description_raw", "item_description_bm25"),
):
    artifact_train_items[target_column] = preprocess_series(train_item_texts[source_column], final_bm25_preprocessing)
artifact_train_items["item_text_bm25"] = (
    artifact_train_items["item_title_bm25"] + " "
    + artifact_train_items["item_params_bm25"] + " "
    + artifact_train_items["item_description_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_train_items["item_title_char_tfidf"] = preprocess_series(
    train_item_texts["item_title_raw"], final_char_preprocessing
)

items_artifact_path = ARTIFACT_DIR / "benchmark_items_text.parquet"
benchmark_queries_artifact_path = ARTIFACT_DIR / "benchmark_queries_text.parquet"
train_queries_artifact_path = ARTIFACT_DIR / "train_query_contexts_text.parquet"
train_items_artifact_path = ARTIFACT_DIR / "train_items_text.parquet"
validation_results_path = ARTIFACT_DIR / "validation_results.csv"
manifest_path = ARTIFACT_DIR / "manifest.json"

artifact_items.to_parquet(items_artifact_path, index=False)
artifact_benchmark_queries.to_parquet(benchmark_queries_artifact_path, index=False)
artifact_train_queries.to_parquet(train_queries_artifact_path, index=False)
artifact_train_items.to_parquet(train_items_artifact_path, index=False)
pd.DataFrame(results).sort_values("recall@50", ascending=False).to_csv(validation_results_path, index=False)

manifest = {
    "artifact_version": PREPROCESSING_VERSION,
    "created_by": "m1_lexical_retrieval.ipynb",
    "final_experiment": str(best_final_result["experiment"]),
    "bm25_preprocessing": final_bm25_preprocessing,
    "char_tfidf_preprocessing": final_char_preprocessing,
    "preprocessing_specs": PREPROCESSING_SPECS,
    "russian_stopword_count": len(RUSSIAN_STOPWORDS),
    "source_rows": {
        "benchmark_items": len(artifact_items),
        "benchmark_queries": len(artifact_benchmark_queries),
        "unique_train_query_contexts": len(artifact_train_queries),
        "unique_train_items": len(artifact_train_items),
    },
    "source_files": {
        path.name: {"bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)
    },
    "files": [
        items_artifact_path.name,
        benchmark_queries_artifact_path.name,
        train_queries_artifact_path.name,
        train_items_artifact_path.name,
        validation_results_path.name,
    ],
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# Lightweight artifact-level verification, not merely a path-exists check.
assert len(pd.read_parquet(items_artifact_path, columns=["item_id", "item_text_bm25"])) == len(candidate_items)
assert len(pd.read_parquet(benchmark_queries_artifact_path, columns=["query_id", "search_text_bm25"])) == len(benchmark_queries)
assert len(pd.read_parquet(train_queries_artifact_path, columns=["query_group", "search_text_bm25"])) == train_pairs["query_group"].nunique()
assert len(pd.read_parquet(train_items_artifact_path, columns=["item_id", "item_text_bm25"])) == len(artifact_train_items)

artifact_seconds = time.perf_counter() - artifact_started
print({
    "artifact_dir": str(ARTIFACT_DIR),
    "artifact_seconds": round(artifact_seconds, 2),
    "final_bm25_preprocessing": final_bm25_preprocessing,
    "final_char_preprocessing": final_char_preprocessing,
    "files": manifest["files"],
})

## 9. Export final top-200 candidate lists

Sklearn matrices are intentionally not serialized: they are large and fragile across package versions. Instead, rebuild only the selected final pair of indices once, then export item IDs for every validation and benchmark query. These compact artifacts can be reused by M2–M6 without re-indexing.

In [ ]:
def allowed_indices_for_queries(query_frame: pd.DataFrame) -> list[np.ndarray]:
    return [allowed_indices_for_category(category)[0] for category in query_frame["search_category"]]


final_validation_rankings_by_experiment = {
    "E03c_rrf_bm25_all_fields_char_tfidf": rankings_rrf_char,
    "E04_rrf_best_bm25_control_char": rankings_rrf_best_bm25_control_char,
    "E04_rrf_best_bm25_best_char": rankings_rrf_best_both,
}
final_validation_rankings = final_validation_rankings_by_experiment[str(best_final_result["experiment"])]

benchmark_query_text = benchmark_queries["search_query"].fillna("").astype(str).tolist()
benchmark_allowed_indices = allowed_indices_for_queries(benchmark_queries)
export_started = time.perf_counter()

# Refit only the two selected final sources. This is the production-like M1
# inference path and is much cheaper than retaining all six ablation indices.
export_bm25, export_bm25_index_seconds = fit_sparse_bm25(all_documents, preprocessing_key=final_bm25_preprocessing)
benchmark_bm25_rankings, export_bm25_retrieval_seconds = retrieve_bm25(
    export_bm25, benchmark_query_text, benchmark_allowed_indices, TOP_K
)
export_char_vectorizer, benchmark_char_rankings, export_char_index_seconds, export_char_retrieval_seconds = fit_and_retrieve_char_tfidf(
    char_documents, benchmark_query_text, benchmark_allowed_indices, preprocessing_key=final_char_preprocessing
)
benchmark_final_rankings = [
    rrf_fuse([bm25_ranking, char_ranking], TOP_K)
    for bm25_ranking, char_ranking in zip(benchmark_bm25_rankings, benchmark_char_rankings, strict=True)
]


def rankings_to_frame(identifier_name: str, identifiers: pd.Series, rankings: list[np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame({
        identifier_name: identifiers.to_numpy(),
        "candidate_item_ids_top200": [" ".join(candidate_item_ids[ranking].tolist()) for ranking in rankings],
    })


validation_top200_path = ARTIFACT_DIR / "m1_validation_top200.parquet"
benchmark_top200_path = ARTIFACT_DIR / "m1_benchmark_top200.parquet"
rankings_to_frame("query_group", validation_queries["query_group"], final_validation_rankings).to_parquet(
    validation_top200_path, index=False
)
rankings_to_frame("query_id", benchmark_queries["query_id"], benchmark_final_rankings).to_parquet(
    benchmark_top200_path, index=False
)

manifest_path = ARTIFACT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["candidate_exports"] = {
    "validation": validation_top200_path.name,
    "benchmark": benchmark_top200_path.name,
    "top_k": TOP_K,
    "final_experiment": str(best_final_result["experiment"]),
    "final_index_timing_seconds": {
        "bm25_build": export_bm25_index_seconds,
        "bm25_retrieval_benchmark": export_bm25_retrieval_seconds,
        "char_build": export_char_index_seconds,
        "char_retrieval_benchmark": export_char_retrieval_seconds,
    },
}
manifest["files"].extend([validation_top200_path.name, benchmark_top200_path.name])
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# A single downloadable bundle contains all reusable data; the full output root
# is still persisted automatically by Kaggle Save & Run All.
bundle_path = Path(shutil.make_archive(
    str(OUTPUT_ROOT / "m1_kaggle_artifacts"),
    "zip",
    root_dir=OUTPUT_ROOT,
    base_dir=ARTIFACT_DIR.relative_to(OUTPUT_ROOT),
))

assert len(pd.read_parquet(validation_top200_path, columns=["query_group"])) == len(validation_queries)
assert len(pd.read_parquet(benchmark_top200_path, columns=["query_id"])) == len(benchmark_queries)
export_seconds = time.perf_counter() - export_started
print({
    "export_seconds": round(export_seconds, 2),
    "final_bm25_preprocessing": final_bm25_preprocessing,
    "final_char_preprocessing": final_char_preprocessing,
    "validation_top200": str(validation_top200_path),
    "benchmark_top200": str(benchmark_top200_path),
    "bundle": str(bundle_path),
})

## 10. Results, ClearML logging and final checks

This cell writes aggregate metrics to the live ClearML task, records the Kaggle export timing and closes the task. The final print is the completion signal to verify before saving the Kaggle version.

In [ ]:
results_frame = pd.DataFrame(results).sort_values(["recall@50", "recall@200", "experiment"], ascending=[False, False, True]).reset_index(drop=True)
display(results_frame)

notebook_seconds = time.perf_counter() - NOTEBOOK_STARTED

for _, result in results_frame.iterrows():
    experiment = str(result["experiment"])
    for k in METRIC_KS:
        clearml_logger.report_scalar(
            title=f"Recall@{k}", series=experiment, value=float(result[f"recall@{k}"]), iteration=0
        )
    clearml_logger.report_scalar(title="HitRate@50", series=experiment, value=float(result["hit_rate@50"]), iteration=0)
    clearml_logger.report_scalar(title="Index seconds", series=experiment, value=float(result["index_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Retrieval seconds", series=experiment, value=float(result["retrieval_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Peak RSS MB", series=experiment, value=float(result["peak_rss_mb"]), iteration=0)

clearml_logger.report_scalar(title="M1 wall time seconds", series="notebook", value=float(notebook_seconds), iteration=0)
clearml_logger.report_scalar(title="Preprocessing artifact seconds", series="notebook", value=float(artifact_seconds), iteration=0)
clearml_logger.report_scalar(title="Final benchmark export seconds", series="notebook", value=float(export_seconds), iteration=0)
clearml_logger.report_scalar(title="Category-114 partition items", series="corpus", value=float((candidate_items["item_category_id"] == 114).sum()), iteration=0)
try:
    clearml_logger.report_table(
        title="M1 validation summary",
        series="E01-E04",
        iteration=0,
        table_plot=results_frame,
    )
    clearml_logger.report_table(
        title="M1 preprocessing ablation",
        series="word_bm25",
        iteration=0,
        table_plot=preprocessing_frame,
    )
    table_log_status = "logged"
except Exception as exc:
    table_log_status = f"table logging skipped: {type(exc).__name__}"

clearml_task.set_parameter("results/best_final_experiment", str(best_final_result["experiment"]))
clearml_task.set_parameter("results/best_final_recall_at_50", float(best_final_result["recall@50"]))
clearml_task.set_parameter("results/best_bm25_preprocessing", final_bm25_preprocessing)
clearml_task.set_parameter("results/best_char_preprocessing", final_char_preprocessing)
clearml_task.set_parameter("artifacts/text_preprocessing_dir", str(ARTIFACT_DIR))
clearml_task.set_parameter("artifacts/benchmark_top200", str(benchmark_top200_path))
clearml_task.set_parameter("artifacts/bundle", str(bundle_path))
clearml_task.set_parameter("results/notebook_seconds", float(notebook_seconds))
clearml_task.close()

print({
    "best_final_experiment": best_final_result["experiment"],
    "best_final_recall@50": round(float(best_final_result["recall@50"]), 6),
    "best_bm25_preprocessing": final_bm25_preprocessing,
    "best_char_preprocessing": final_char_preprocessing,
    "notebook_seconds": round(notebook_seconds, 2),
    "clearml_table_status": table_log_status,
    "kaggle_output_bundle": str(bundle_path),
})

## Completion and artifact locations

A successful run writes all reusable files below `/kaggle/working/avito-m1-lexical/`. Use **Save Version → Save & Run All** so Kaggle snapshots the code, logs, inputs and output files. In the completed version, open **Output** to download `m1_kaggle_artifacts.zip`, or attach this Notebook Output as an Input to a later Kaggle notebook.

The selection rule remains metric-driven: use the executed results table, not a pre-written conclusion, as the source of truth.